<a href="https://colab.research.google.com/github/2025ct05044-cmd/MLOps-Assignment/blob/main/Notebook/04_FastAPI_Model_Serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — FastAPI Model Serving
**MLOps Assignment 01: Heart Disease Prediction**

This notebook checks the project files, installs API dependencies, validates the saved model, tests the API request schema, and starts FastAPI with Uvicorn.

> **Important:** This is an educational machine-learning demonstration, not a medical diagnostic tool. Predictions must not be used for medical decisions.

## Prerequisites
- Open this notebook in the same Colab runtime where the repository is available at `/content/MLOps-Assignment`.
- The repository should contain `api/app.py` and `models/heart_disease_pipeline.joblib`.
- Run cells in order.

## 1. Set the project directory and verify files

In [45]:
%cd /content

!git clone https://github.com/2025ct05044-cmd/MLOps-Assignment.git

/content
fatal: destination path 'MLOps-Assignment' already exists and is not an empty directory.


In [46]:
%cd /content/MLOps-Assignment

!pwd
!ls -lh api/
!ls -lh models/heart_disease_pipeline.joblib
!ls -lh requirements.txt Dockerfile

/content/MLOps-Assignment
/content/MLOps-Assignment
total 12K
-rw-r--r-- 1 root root 2.3K Oct 19 14:04 app.py
-rw-r--r-- 1 root root   60 Oct 19 14:04 __init__.py
drwxr-xr-x 2 root root 4.0K Oct 19 14:05 __pycache__
-rw-r--r-- 1 root root 4.9K Oct 19 14:04 models/heart_disease_pipeline.joblib
-rw-r--r-- 1 root root 481 Oct 19 14:04 Dockerfile
-rw-r--r-- 1 root root 163 Oct 19 14:04 requirements.txt


In [47]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/content/MLOps-Assignment")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}. "
        "Clone or upload your repository to this location first."
    )

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

required_files = [
    Path("api/app.py"),
    Path("api/__init__.py"),
    Path("models/heart_disease_pipeline.joblib"),
    Path("requirements.txt"),
    Path("Dockerfile"),
]
for file_path in required_files:
    print(f"{'OK' if file_path.is_file() else 'MISSING'}: {file_path}")

missing = [str(p) for p in required_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Required project files are missing: " + ", ".join(missing))

Working directory: /content/MLOps-Assignment
OK: api/app.py
OK: api/__init__.py
OK: models/heart_disease_pipeline.joblib
OK: requirements.txt
OK: Dockerfile


## 2. Install dependencies

In [48]:
%pip install -r requirements.txt
%pip install fastapi uvicorn

## 3. Verify imports and load the trained pipeline

In [49]:
import fastapi
import uvicorn
import joblib
import pandas as pd

MODEL_PATH = PROJECT_ROOT / "models" / "heart_disease_pipeline.joblib"
model = joblib.load(MODEL_PATH)

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("Pandas:", pd.__version__)
print("Model type:", type(model).__name__)
print("Pipeline steps:", getattr(model, "named_steps", {}).keys())
print("All required imports and model loading successful.")

FastAPI: 0.141.1
Uvicorn: 0.54.0
Pandas: 2.2.3
Model type: Pipeline
Pipeline steps: dict_keys(['preprocessor', 'classifier'])
All required imports and model loading successful.


## 4. Inspect the API routes

In [50]:
from api.app import app, FEATURE_COLUMNS, MODEL_PATH as API_MODEL_PATH

print("API title:", app.title)
print("API version:", app.version)
print("API model path:", API_MODEL_PATH)
print("Number of input features:", len(FEATURE_COLUMNS))
print("Feature order:", FEATURE_COLUMNS)
print("\nRegistered routes:")
for route in app.routes:
    methods = ",".join(sorted(getattr(route, "methods", []) or []))
    print(f"{methods:12s} {route.path}")

API title: Heart Disease Prediction API
API version: 1.0.0
API model path: /content/MLOps-Assignment/models/heart_disease_pipeline.joblib
Number of input features: 13
Feature order: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

Registered routes:
GET,HEAD     /openapi.json
GET,HEAD     /docs
GET,HEAD     /docs/oauth2-redirect
GET,HEAD     /redoc
GET          /
GET          /health
POST         /predict


## 5. Test API validation and prediction in-process

This cell uses FastAPI's `TestClient` to test the API without needing a separate server process. It checks the root and health endpoints, submits a sample payload to `/predict`, and checks that missing fields are rejected.

The sample values are for software testing only; they are **not** a real person's measurements and must not be interpreted medically.

In [51]:
from fastapi.testclient import TestClient

sample_payload = {
    "age": 54,
    "sex": 1,
    "cp": 0,
    "trestbps": 130,
    "chol": 250,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 1,
    "ca": 0,
    "thal": 2,
}

with TestClient(app) as client:
    root_response = client.get("/")
    print("GET / status:", root_response.status_code)
    print("GET / response:", root_response.json())

    health_response = client.get("/health")
    print("\nGET /health status:", health_response.status_code)
    print("GET /health response:", health_response.json())

    prediction_response = client.post("/predict", json=sample_payload)
    print("\nPOST /predict status:", prediction_response.status_code)
    print("POST /predict response:", prediction_response.json())

    invalid_response = client.post("/predict", json={"age": 54})
    print("\nInvalid request status (expected 422):", invalid_response.status_code)
    assert root_response.status_code == 200
    assert health_response.status_code == 200
    assert prediction_response.status_code == 200
    assert invalid_response.status_code == 422
    prediction_result = prediction_response.json()
    assert prediction_result["prediction"] in (0, 1)
    assert 0.0 <= prediction_result["probability_class_1"] <= 1.0

print("\nPASS: API endpoints and input validation work.")

GET / status: 200
GET / response: {'message': 'Heart Disease Prediction API is running', 'health': '/health', 'interactive_docs': '/docs'}

GET /health status: 200
GET /health response: {'status': 'ok', 'model_loaded': True}

POST /predict status: 200
POST /predict response: {'prediction': 0, 'probability_class_1': 0.203998931490761, 'model': 'heart_disease_pipeline', 'note': 'For educational use only; not a medical diagnosis.'}

Invalid request status (expected 422): 422

PASS: API endpoints and input validation work.


## 6. Start the API server (optional)

Run this cell only if you want to start Uvicorn in the current Colab runtime. It runs in the foreground, so the cell will remain busy until you interrupt it. To avoid that, use the background option in the next cell instead.

**Do not run both server-start cells at the same time.**

In [52]:
#!uvicorn api.app:app --host 0.0.0.0 --port 8000

## 7. Optional: start Uvicorn in the background and check health

In [53]:
import subprocess
import time
import urllib.request
import urllib.error

# Stop a previously started server from this notebook, if any.
try:
    if "uvicorn_process" in globals() and uvicorn_process.poll() is None:
        uvicorn_process.terminate()
        uvicorn_process.wait(timeout=5)
except Exception as exc:
    print("Previous server cleanup note:", exc)

uvicorn_log_path = PROJECT_ROOT / "reports" / "uvicorn.log"
uvicorn_log_path.parent.mkdir(parents=True, exist_ok=True)
uvicorn_log = open(uvicorn_log_path, "w", encoding="utf-8")

uvicorn_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "api.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=PROJECT_ROOT,
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
)

health_url = "http://127.0.0.1:8000/health"
for attempt in range(20):
    if uvicorn_process.poll() is not None:
        raise RuntimeError(
            "Uvicorn exited early. Check reports/uvicorn.log:\n"
            + uvicorn_log_path.read_text(encoding="utf-8")
        )
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            print("Health HTTP status:", response.status)
            print("Health response:", response.read().decode("utf-8"))
            break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError(
        "API did not become ready. Check reports/uvicorn.log:\n"
        + uvicorn_log_path.read_text(encoding="utf-8")
    )

print("API docs URL (inside this runtime): http://127.0.0.1:8000/docs")
print("Server PID:", uvicorn_process.pid)

Health HTTP status: 200
Health response: {"status":"ok","model_loaded":true}
API docs URL (inside this runtime): http://127.0.0.1:8000/docs
Server PID: 8523


## 8. Send a live prediction request

In [54]:
import json
import urllib.request

request = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(sample_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(request, timeout=10) as response:
    print("HTTP status:", response.status)
    print(json.dumps(json.loads(response.read().decode("utf-8")), indent=2))

HTTP status: 200
{
  "prediction": 0,
  "probability_class_1": 0.203998931490761,
  "model": "heart_disease_pipeline",
  "note": "For educational use only; not a medical diagnosis."
}


## 9. View server logs if something fails

In [55]:
log_file = PROJECT_ROOT / "reports" / "uvicorn.log"
if log_file.exists():
    print(log_file.read_text(encoding="utf-8")[-12000:])
else:
    print("No Uvicorn log found yet. Run the background server cell first.")

INFO:     Started server process [8523]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:35316 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:35320 - "POST /predict HTTP/1.1" 200 OK



In [56]:
from google.colab import output

output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 10. Stop the Background Server

Run the cell below to manage the API server.

Enter stop to stop the server.

Press Enter to keep it running.

The server automatically stops after 10 minutes.

You can also interrupt the cell using Colab's stop button.

Keep the Colab runtime active while testing the API.

In [57]:
import subprocess
import time

MAX_WAIT_SECONDS = 10 * 60  # 10 minutes

if "uvicorn_process" in globals() and uvicorn_process.poll() is None:
    print("Uvicorn server is running.")
    print("Enter 'stop' to stop the server.")
    print("The server will automatically stop after 10 minutes.")

    start_time = time.monotonic()

    try:
        while uvicorn_process.poll() is None:
            elapsed = time.monotonic() - start_time
            remaining = MAX_WAIT_SECONDS - elapsed

            if remaining <= 0:
                print("\n10-minute timeout reached. Stopping server.")
                break

            try:
                user_input = input(
                    f"\nServer is running. Time remaining: "
                    f"{int(remaining)} seconds. Enter 'stop' to stop it, "
                    "or press Enter to continue: "
                ).strip().lower()

                if user_input == "stop":
                    print("Stop command received.")
                    break

            except EOFError:
                print("\nInput unavailable. Server will stop at timeout.")
                time.sleep(remaining)
                break

    except KeyboardInterrupt:
        print("\nKeyboard interrupt received. Stopping server.")

    finally:
        if uvicorn_process.poll() is None:
            uvicorn_process.terminate()

            try:
                uvicorn_process.wait(timeout=5)
                print("Uvicorn server stopped.")
            except subprocess.TimeoutExpired:
                uvicorn_process.kill()
                uvicorn_process.wait()
                print("Uvicorn server force-stopped.")

else:
    print("No running Uvicorn process found.")

try:
    if "uvicorn_log" in globals() and not uvicorn_log.closed:
        uvicorn_log.close()
        print("Server log file closed.")
except Exception as exc:
    print("Could not close the log file:", exc)

Uvicorn server is running.
Enter 'stop' to stop the server.
The server will automatically stop after 10 minutes.

Server is running. Time remaining: 599 seconds. Enter 'stop' to stop it, or press Enter to continue: 

Server is running. Time remaining: 581 seconds. Enter 'stop' to stop it, or press Enter to continue: stop
Stop command received.
Uvicorn server stopped.
Server log file closed.
